In [0]:
%sql
CREATE DATABASE IF NOT EXISTS data_dev.datadb;


In [0]:
%sql
create volume IF NOT EXISTS data_dev.datadb.files

## DataFrames

In [0]:
df_csv=spark.read.csv("/Volumes/data_dev/datadb/files/orders/Orders.csv",header=True,inferSchema=True)

In [0]:
display(df_csv)

In [0]:
df_csv=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/data_dev/datadb/files/orders/Orders.csv")

df_csv=spark.read.format("csv").options(header=True,inferSchema=True).load("/Volumes/data_dev/datadb/files/orders/Orders.csv")

## JSON or Parquet

In [0]:
df_json=spark.read.json("/Volumes/data_dev/datadb/files/employee/employees.json",multiLine=True)
display(df_json)

In [0]:
df_json=spark.read.format("json").option("multiLine","true").load("/Volumes/data_dev/datadb/files/employee/employees.json")
display(df_json)

In [0]:
df_parquet=spark.read.parquet("/Volumes/data_dev/datadb/files/employee/employees.parquet")
display(df_parquet)

In [0]:
df=spark.read.format("parquet").load("/Volumes/data_dev/datadb/files/employee/employees.parquet")
display(df)

## Select Transformation

In [0]:
df_parquet.select("employee_id","first_name","last_name","salary")
display(df_parquet.select("employee_id","first_name","last_name","salary"))

In [0]:
df_parquet.select(df_parquet["employee_id"],df_parquet["first_name"],df_parquet["last_name"],df_parquet["salary"])

In [0]:
df_parquet.select(df_parquet.employee_id,df_parquet.first_name,df_parquet.last_name,df_parquet.salary)


In [0]:
#when ever we have aggreation methods col functions helps a lot.
from pyspark.sql.functions import *
df_parquet.select(col("employee_id"),col("first_name"),col("last_name"),col("salary").alias("salary in $"))


## withColoumn and withColumnRenamed
###withColumn is used to create new column ex:(salary+100)

In [0]:
df_parquet.withColumn("employee+100",col("employee_id")+100)

display(df_parquet.withColumn("employee+100",col("employee_id")+100))

In [0]:
display(df_parquet.withColumn("Country",lit("India")))
df_parquet.withColumn("salary",col("salary")*10)

In [0]:
display(df_parquet.withColumnRenamed("employee_id","eid"))

#UDF

In [0]:
data = [
  ("Dileep","Bangalore",40),
  ("ashu","Bangalore",38),
  ("charlie","Pune",60),
  ("David","Chennai",18),
  ("Karna","Indore",25),
  ("Alice","Delhi",10),
  ("bob","Mumbai",None)
]

columns = ["Name" , "City" , "Age"]

df=spark.createDataFrame(data,columns)
display(df)

In [0]:
def age_group(age):
  if age is None:
    return "Unknown"
  elif age < 18:
    return "Teenager"
  elif age >=18 and age < 40:
    return "Adult"
  else:
    return "Senior Citizen"
  
age_group(10)
age_group(20)
age_group(50)


In [0]:
# To test Function
print(age_group(10))
print(age_group(20))
print(age_group(50))

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
age_group_udf = udf(age_group,StringType()) # Without return type also we can register the function.
df.withColumn("Age Group",age_group_udf(col("Age"))).show()
# df.withColumn("Age",age_group_udf(df.Age")).show()